# Bronze: load the mandi files

Loads the CSV files that match `file_path` into `bronze.raw_mandi_transactions` in the workspace's default catalog
(the table is created by `src/setup`). Each file is loaded separately.

- Source columns are loaded exactly as received (all STRING, no trimming or cleaning).
- Metadata columns are derived here: `_ingested_at`, `_source_mode` (`FILE`), `_file_name`, `_run_id`, `_row_no`.
- Idempotent per run: loading the same file again with the same `run_id` replaces its rows; other runs and files stay. Duplicates between runs are removed in silver.
- The files stay where they are; the last task of `mandi_daily` (`src/archive_files`) moves them to the archive.

Parameters (both required):
- `file_path`: path of one CSV file or a pattern, e.g. `/Volumes/<catalog>/bronze/landing_zone/mandi_*.csv`.
- `run_id`: id of the run loading the files.

If a load fails or no file matches, the reason is printed and the notebook still completes (it fails when `fail_on_error` is `true`).

In [ ]:
dbutils.widgets.text("file_path", "", "File path")
dbutils.widgets.text("run_id", "", "Run id")
dbutils.widgets.text("fail_on_error", "false", "Fail the job on error (true/false)")

In [ ]:
import os

from lib_mandi.config import OPTIONS_READ_SOURCE, SCHEMA_SOURCE
from lib_mandi.extract import extract__source_file, extract__source_paths
from lib_mandi.load import load__bronze
from lib_mandi.transformations import transform__bronze
from lib_mandi.utility import get__default_catalog

file_path = dbutils.widgets.get("file_path").strip()
run_id = dbutils.widgets.get("run_id").strip()

if not file_path or not run_id:
    raise ValueError("Widgets 'file_path' and 'run_id' are required")

catalog = get__default_catalog(spark)
if not catalog:
    raise RuntimeError("Could not read the default catalog (see the error printed above)")

paths = extract__source_paths(file_path)
loaded = paths is not None
for path in paths or []:
    file_name = os.path.basename(path)
    df = extract__source_file(spark, path, SCHEMA_SOURCE, OPTIONS_READ_SOURCE)
    if df is not None:
        df = transform__bronze(df, run_id, file_name)
    if df is None or not load__bronze(df, f"{catalog}.bronze.raw_mandi_transactions", run_id, file_name):
        loaded = False

if not loaded and dbutils.widgets.get("fail_on_error").strip().lower() == "true":
    raise RuntimeError("Load did not finish (see the reason printed above)")